# 013: Prompt Enhancement A/B Test — Base Rate Reasoning

Tests whether enhanced prompt wording improves base rate reasoning quality.
Compares the current generic "Consider base rates and analogs" prompt section
against an enhanced version with Fermi decomposition steps and explicit reference
class identification.

**Production target**: `main.py` — enhanced prompt text for the "Consider base rates and analogs" sections (~lines 254, 562, 713)

**Functions to extract**: None (prompt text only)

**Success criteria**: Enhanced prompts produce more structured base rate reasoning in at least 60% of test questions.

## Update Procedure

| Component | Source File | Lines |
|---|---|---|
| Base rate prompt (binary) | `main.py` | ~254 |
| Base rate prompt (MC) | `main.py` | ~562 |
| Base rate prompt (numeric) | `main.py` | ~713 |
| LLM config | `main.py` | ~1362 |

In [ ]:
# EXPLORATION ONLY
# Cell 1: Setup & Imports

import os
import json
import asyncio
import time
from types import SimpleNamespace

import requests
import nest_asyncio
from dotenv import load_dotenv
from forecasting_tools import GeneralLlm

nest_asyncio.apply()
load_dotenv()

# --- LLM Configuration (match main.py llms dict) ---
DEFAULT_MODEL = "openrouter/openai/gpt-5.2"  # temperature=1

print(f"Default model: {DEFAULT_MODEL}")
print(f"OPENROUTER_API_KEY set: {'OPENROUTER_API_KEY' in os.environ}")

In [ ]:
# EXPLORATION ONLY
# Cell 2: Define Test Questions
# Using SimpleNamespace so prompt templates work unchanged (same attribute names as bot framework)

# Fetch a question from Metaculus API
def fetch_metaculus_question(question_id):
    """Fetch question data from Metaculus API and return as SimpleNamespace."""
    url = f"https://www.metaculus.com/api/questions/{question_id}/"
    headers = {"Authorization": f"Token {os.environ.get('METACULUS_BOT_API_TOKEN', '')}"}
    resp = requests.get(url, headers=headers)
    resp.raise_for_status()
    data = resp.json()
    return SimpleNamespace(
        id_of_question=data.get("id"),
        question_text=data.get("title", ""),
        resolution_criteria=data.get("resolution_criteria", ""),
        fine_print=data.get("fine_print", ""),
        background_info=data.get("description", ""),
        question_type=data.get("type", "binary"),
        community_prediction=data.get("community_prediction", {}).get("full", {}).get("q2"),
    )

# --- Test question IDs ---
# Replace with actual question IDs from Metaculus
BINARY_IDS = [42562]     # Example: Ducks playoffs
NUMERIC_IDS = []         # Add numeric question IDs
MC_IDS = []              # Add multiple choice question IDs

all_ids = BINARY_IDS + NUMERIC_IDS + MC_IDS
test_questions = []
for qid in all_ids:
    try:
        q = fetch_metaculus_question(qid)
        test_questions.append(q)
        print(f"Loaded Q{q.id_of_question}: {q.question_text[:80]}...")
    except Exception as e:
        print(f"Failed to load Q{qid}: {e}")

print(f"\nTotal test questions: {len(test_questions)}")

In [ ]:
# EXPLORATION ONLY
# Cell 3: Load News/Research from External File
# Follows established pattern from 012 series — research provided as external text file

NEWS_FILE = "../data/Run Log News Summaries/42562_News_Summary_03-15-2026.txt"

with open(NEWS_FILE, encoding='utf-8') as f:
    news_summary = f.read()

# Alias for prompt compatibility
research = news_summary

print(f"Loaded: {NEWS_FILE}")
print(f"Length: {len(research)} characters")
print(f"Preview: {research[:200]}...")

In [ ]:
# EXPLORATION ONLY
# Cell 4: Define Prompt Variants

# --- BASELINE: Current prompt section from main.py (~line 254) ---
BASELINE_BASE_RATE_SECTION = """\
### Consider base rates and analogs
- Are there analogs that suggest what the probability should be in the absence of other evidence (base rate)
- Could this be a question dominated by simple probability, e.g. the chance that the roll of a single dice might be 6
- How should base rates anchor or adjust your interpretation of the scenario range?
- Note your observations on base rates
"""

# --- ENHANCED: Structured Fermi decomposition + explicit reference class ---
ENHANCED_BASE_RATE_SECTION = """\
### Consider base rates and analogs
Before forming your forecast, ground your reasoning in historical frequencies:

1. **Identify the reference class**: What is the most relevant category of similar past events?
   - State the reference class explicitly (e.g., "US midterm elections where the incumbent party lost the House")
   - If multiple reference classes apply, list the top 2-3 and note which is most specific

2. **Estimate the base rate**: Out of all instances in your reference class, how often did the outcome of interest occur?
   - State as a fraction: approximately [X] out of [Y] times (numerator / denominator)
   - If you cannot find an exact count, provide your best Fermi estimate with reasoning

3. **Assess trend direction**: Is the base rate increasing, decreasing, or stable over recent periods?
   - Note any structural changes that make recent data more/less relevant

4. **Apply the base rate as your anchor**: Start from this base rate, then adjust up or down based on
   question-specific evidence. State how far and why you are adjusting from the base rate.

5. **Flag if no base rate applies**: If this is a genuinely novel event with no historical precedent,
   state that explicitly rather than inventing a spurious reference class.
"""

print("=== BASELINE ===")
print(BASELINE_BASE_RATE_SECTION)
print("\n=== ENHANCED ===")
print(ENHANCED_BASE_RATE_SECTION)

In [ ]:
# EXPLORATION ONLY
# Cell 5: Build Full Forecast Prompts (Binary variant shown)
# Embeds each base rate section variant into the binary forecast prompt skeleton

BINARY_PROMPT_TEMPLATE = """\
You are a professional forecaster interviewing for a job.

Your research assistant says:
{research}

Question: {question_text}
Resolution criteria: {resolution_criteria}
Fine print: {fine_print}
Background: {background_info}

### Before answering you write:
1. The time left until the outcome to the question is known.
2. The status quo outcome if nothing changed.
3. The expectations of experts and markets.
4. A brief description of a scenario that results in a No outcome.
5. A brief description of a scenario that results in a Yes outcome.

### You write your rationale remembering that good forecasters put extra weight on the status quo outcome
since the world changes slowly most of the time.

{base_rate_section}

After writing your rationale, provide your forecast as a probability between 1% and 99%.
Format: **Probability: XX%**
"""

def build_forecast_prompt(question, research_text, base_rate_section):
    return BINARY_PROMPT_TEMPLATE.format(
        research=research_text,
        question_text=question.question_text,
        resolution_criteria=getattr(question, 'resolution_criteria', ''),
        fine_print=getattr(question, 'fine_print', ''),
        background_info=getattr(question, 'background_info', ''),
        base_rate_section=base_rate_section,
    )

# Preview
if test_questions:
    sample = build_forecast_prompt(test_questions[0], research[:500], ENHANCED_BASE_RATE_SECTION)
    print(f"Prompt length: {len(sample)} chars")
    print(f"Preview (last 400 chars):\n...{sample[-400:]}")

In [ ]:
# EXPLORATION ONLY
# Cell 6: Run A/B Test — Baseline vs Enhanced

import re

async def run_forecast(question, research_text, base_rate_section, model=DEFAULT_MODEL, temperature=1.0):
    """Run a single forecast and return the response + extracted probability."""
    prompt = build_forecast_prompt(question, research_text, base_rate_section)
    llm = GeneralLlm(model=model, temperature=temperature, timeout=80)
    
    start = time.time()
    response = await llm.invoke(prompt)
    elapsed = time.time() - start
    
    # Extract probability from response
    prob_match = re.search(r'\*\*Probability:\s*([\d.]+)%\*\*', response)
    prob = float(prob_match.group(1)) if prob_match else None
    
    return {
        'response': response,
        'probability': prob,
        'elapsed_s': elapsed,
        'prompt_chars': len(prompt),
    }

# Run both variants for each test question
results = []
for q in test_questions:
    print(f"\n--- Q{q.id_of_question}: {q.question_text[:60]}... ---")
    
    baseline_result = await run_forecast(q, research, BASELINE_BASE_RATE_SECTION)
    print(f"  Baseline: {baseline_result['probability']}% ({baseline_result['elapsed_s']:.1f}s)")
    
    enhanced_result = await run_forecast(q, research, ENHANCED_BASE_RATE_SECTION)
    print(f"  Enhanced: {enhanced_result['probability']}% ({enhanced_result['elapsed_s']:.1f}s)")
    
    results.append({
        'question_id': q.id_of_question,
        'question_text': q.question_text[:80],
        'community': q.community_prediction,
        'baseline': baseline_result,
        'enhanced': enhanced_result,
    })

print(f"\nCompleted {len(results)} A/B comparisons.")

In [ ]:
# EXPLORATION ONLY
# Cell 7: Display Results — Side-by-Side Comparison

print(f"{'Q ID':<8} {'Baseline':>10} {'Enhanced':>10} {'Community':>10} {'Question':<50}")
print("-" * 95)

for r in results:
    bp = r['baseline']['probability']
    ep = r['enhanced']['probability']
    cp = r['community']
    bp_str = f"{bp:.1f}%" if bp else "N/A"
    ep_str = f"{ep:.1f}%" if ep else "N/A"
    cp_str = f"{cp*100:.1f}%" if cp else "N/A"
    print(f"{r['question_id']:<8} {bp_str:>10} {ep_str:>10} {cp_str:>10} {r['question_text']:<50}")

print("\n--- Base Rate Reasoning Quality (Manual Scoring) ---")
print("Review the full responses below and score each 1-5:")
print("  1 = No base rate mentioned")
print("  2 = Generic mention without numbers")
print("  3 = Reference class identified, rough estimate given")
print("  4 = Explicit numerator/denominator, trend noted")
print("  5 = Full Fermi decomposition with cited reasoning")

In [ ]:
# EXPLORATION ONLY
# Cell 8: Full Response Comparison (for manual scoring)

for r in results:
    print(f"\n{'='*80}")
    print(f"Q{r['question_id']}: {r['question_text']}")
    print(f"Community: {r['community']}")
    
    print(f"\n--- BASELINE RESPONSE (prob={r['baseline']['probability']}%) ---")
    print(r['baseline']['response'])
    
    print(f"\n--- ENHANCED RESPONSE (prob={r['enhanced']['probability']}%) ---")
    print(r['enhanced']['response'])
    print()

In [ ]:
# EXPLORATION ONLY
# Cell 9: Cost Estimation
# Estimates OpenRouter API cost per model based on approximate token counts

# Approximate pricing (per 1M tokens, as of March 2026)
MODEL_PRICING = {
    "openrouter/openai/gpt-5.2": {"input": 2.00, "output": 8.00},
    "openrouter/openai/gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "openrouter/openai/o4-mini": {"input": 1.10, "output": 4.40},
}

# Rough token estimate: 1 token ~ 4 chars
CHARS_PER_TOKEN = 4

total_cost = 0.0
print(f"{'Variant':<12} {'Q ID':<8} {'Input Tok':>10} {'Output Tok':>11} {'Cost ($)':>10}")
print("-" * 55)

pricing = MODEL_PRICING.get(DEFAULT_MODEL, {"input": 2.00, "output": 8.00})

for r in results:
    for variant in ['baseline', 'enhanced']:
        data = r[variant]
        input_tokens = data['prompt_chars'] / CHARS_PER_TOKEN
        output_tokens = len(data['response']) / CHARS_PER_TOKEN
        cost = (input_tokens * pricing['input'] + output_tokens * pricing['output']) / 1_000_000
        total_cost += cost
        print(f"{variant:<12} {r['question_id']:<8} {input_tokens:>10.0f} {output_tokens:>11.0f} {cost:>10.4f}")

print(f"\n{'TOTAL':>42} ${total_cost:.4f}")
print(f"\nModel: {DEFAULT_MODEL}")
print(f"Pricing: ${pricing['input']}/M input, ${pricing['output']}/M output")
print("Note: Token counts are approximate (chars/4). Actual costs may differ.")

## Summary & Next Steps

**If enhanced prompt wins (>60% of questions show better reasoning):**
- Copy `ENHANCED_BASE_RATE_SECTION` into `main.py` at:
  - Binary prompt (~line 254)
  - Multiple choice prompt (~line 562)
  - Numeric prompt (~line 713)

**If baseline wins or tie:**
- Keep current prompt; the value-add may come from *data* (base rate researcher) rather than *prompting*

**Either way:** Proceed to Notebook 013a (Multi-Model Base Rate Researcher) for the real data pipeline.